# AI Roots — Diagnóstico Arquitetural de Redes Neurais

**Episódio:** Arquitetura de Redes Neurais: Neurônios, Camadas e Decisão Baseada em Problema  
**Canal:** [AI Roots](https://youtube.com/@airoots)  

---

Este notebook implementa os três experimentos do episódio:

- **Experimento 1:** Diagnóstico de capacidade — underfitting vs overfitting estrutural
- **Experimento 2:** Fórmula de parâmetros `P = Σ(n_{l−1} · n_l + n_l)` de forma auditável por camada
- **Experimento 3:** Inductive bias — como o alinhamento arquitetura/problema determina a generalização

> **Dataset usado:** `sklearn.datasets.load_digits` (8×8, 10 classes, 1.797 amostras)  
> Rodável offline sem GPU. Substitua por Fashion-MNIST se quiser mais escala.

---

## Setup e dependências

In [1]:
# Se ainda não instalado:
%pip install torch torchinfo matplotlib seaborn scikit-learn --break-system-packages -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torchinfo import summary

warnings.filterwarnings('ignore')

# Reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

## Estilo dos gráficos

In [ ]:
# Paleta AI Roots — mesma do widget interativo do canal
COLORS = {
    'mlp_raso':  '#3266ad',
    'mlp_medio': '#c0592a',
    'mlp_largo': '#2a8a5e',
}

plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#fafafa',
    'axes.grid':         True,
    'grid.alpha':        0.25,
    'grid.linewidth':    0.6,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titlesize':    12,
    'axes.titleweight':  'semibold',
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   9,
    'legend.framealpha': 0.85,
    'font.family':       'sans-serif',
})

---

## Dataset — sklearn Digits

Imagens 8×8 (64 features) de dígitos escritos à mão, 10 classes.  
Simples o suficiente para rodar em CPU em minutos, rico o suficiente para mostrar diferença arquitetural.

In [ ]:
digits = load_digits()
X_raw, y_raw = digits.data.astype('float32'), digits.target

# Normalização: média 0, desvio 1 por feature
# Por que StandardScaler e não MinMaxScaler?
# Porque gradient descent converge mais rápido quando as features
# têm a mesma escala e distribuição centrada no zero.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_scaled, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# Tensores PyTorch
X_tr_t  = torch.tensor(X_tr).to(device)
y_tr_t  = torch.tensor(y_tr, dtype=torch.long).to(device)
X_val_t = torch.tensor(X_val).to(device)
y_val_t = torch.tensor(y_val, dtype=torch.long).to(device)

INPUT_DIM = X_tr_t.shape[1]  # 64
N_CLASSES  = len(set(y_raw))  # 10

print(f'Treino:    {X_tr_t.shape}  |  classes: {N_CLASSES}')
print(f'Validação: {X_val_t.shape}')

# Visualizar algumas amostras
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='Blues', interpolation='nearest')
    ax.set_title(f'classe {digits.target[i]}', fontsize=8)
    ax.axis('off')
plt.suptitle('Amostras do dataset — sklearn Digits (8×8)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

---

## Experimento 1 — Diagnóstico de Capacidade

### Framework: capacidade, inductive bias, landscape

Toda decisão arquitetural define três dimensões simultaneamente:

| Dimensão | O que define | Sintoma de erro |
|---|---|---|
| **Capacidade** | Quais funções a rede *pode* representar | Erro de treino alto → underfitting |
| **Inductive bias** | Quais funções ela *prefere* representar | Gap treino/val crescente → memorização |
| **Landscape** | Quão difícil é otimizar | Curvas instáveis, convergência lenta |

A fórmula de parâmetros treináveis de uma rede densa:

$$P = \sum_{l=1}^{L} \left( n_{l-1} \cdot n_l + n_l \right)$$

onde $n_l$ é o número de unidades na camada $l$ e o segundo termo é o bias.

In [ ]:
# ─── Definição das arquiteturas ───────────────────────────────────────────────
# Cada modelo é uma HIPÓTESE explícita sobre o problema.
# Não são três tentativas aleatórias — são três pontos diferentes
# no trade-off capacidade / inductive bias.

def build_mlp_raso(input_dim=64, n_classes=10):
    """
    Hipótese: o problema é quase linearmente separável.
    Capacidade mínima — serve como baseline de underfitting.
    Se este modelo performa bem, a arquitetura mais complexa
    provavelmente está memorando ruído.
    """
    return nn.Sequential(
        nn.Linear(input_dim, 32), nn.ReLU(),
        nn.Linear(32, n_classes)
    )

def build_mlp_medio(input_dim=64, n_classes=10):
    """
    Hipótese: interações de média complexidade, sem estrutura específica.
    Capacidade adequada para a maioria dos problemas de classificação
    com features de baixa dimensão.
    """
    return nn.Sequential(
        nn.Linear(input_dim, 256), nn.ReLU(),
        nn.Linear(256, 128),       nn.ReLU(),
        nn.Linear(128, n_classes)
    )

def build_mlp_largo(input_dim=64, n_classes=10):
    """
    Hipótese: interações densas e complexas em alta dimensão.
    Capacidade alta com Dropout para regularização implícita.
    Para 64 features e ~1.400 amostras, testa o limite de
    overparameterização benigna vs overfitting clássico.
    """
    return nn.Sequential(
        nn.Linear(input_dim, 512), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(512, 256),       nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, n_classes)
    )

architectures = {
    'MLP Raso':  build_mlp_raso,
    'MLP Médio': build_mlp_medio,
    'MLP Largo': build_mlp_largo,
}

# ─── Auditoria de capacidade (torchinfo) ──────────────────────────────────────
# Auditar ANTES de treinar — não depois.
# Ver onde os parâmetros estão concentrados muda a pergunta de
# "funciona?" para "por quê?"

print('=' * 60)
print('AUDITORIA DE CAPACIDADE — parâmetros treináveis por camada')
print('=' * 60)

for name, builder in architectures.items():
    model = builder(INPUT_DIM, N_CLASSES)
    total = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n{name} — {total:,} parâmetros')
    for i, (layer_name, layer) in enumerate(model.named_modules()):
        if hasattr(layer, 'weight'):
            p = layer.weight.numel() + (layer.bias.numel() if layer.bias is not None else 0)
            print(f'  {layer_name}: {p:>8,} params  |  shape {list(layer.weight.shape)}')

In [ ]:
# ─── Loop de treino ───────────────────────────────────────────────────────────

def train_full(
    model, X_tr, y_tr, X_val, y_val,
    epochs=30, lr=1e-3, batch_size=128
):
    """
    Treina um modelo e devolve o histórico completo.
    Retorna dict com listas de métricas por época.

    Por que batch_size=128 e não full-batch?
    SGD mini-batch tem regularização implícita: o gradiente ruidoso
    favorece mínimos mais planos (flat minima), que tendem a
    generalizar melhor. Com full-batch (gradiente exato), o modelo
    converge para mínimos mais afiados e com pior generalização.
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    n = len(X_tr)
    history = {'tr_loss': [], 'val_loss': [], 'tr_acc': [], 'val_acc': [], 'gap': []}

    for epoch in range(epochs):
        # ── Mini-batch shuffle ──
        perm = torch.randperm(n)
        epoch_loss, epoch_correct = 0.0, 0

        model.train()
        for start in range(0, n, batch_size):
            idx    = perm[start:start + batch_size]
            Xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            out  = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            epoch_loss    += loss.item() * len(idx)
            epoch_correct += (out.argmax(1) == yb).sum().item()

        tr_loss = epoch_loss / n
        tr_acc  = epoch_correct / n

        # ── Validação ──
        model.eval()
        with torch.no_grad():
            val_out  = model(X_val)
            val_loss = criterion(val_out, y_val).item()
            val_acc  = (val_out.argmax(1) == y_val).float().mean().item()

        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['tr_acc'].append(tr_acc)
        history['val_acc'].append(val_acc)
        history['gap'].append(val_loss - tr_loss)  # gap positivo = val > treino (overfitting)

    return history

# ─── Treinar os três modelos ──────────────────────────────────────────────────
EPOCHS = 40
results = {}
param_counts = {}

for name, builder in architectures.items():
    print(f'Treinando: {name}...', end=' ', flush=True)
    model = builder(INPUT_DIM, N_CLASSES)
    param_counts[name] = sum(p.numel() for p in model.parameters() if p.requires_grad)
    results[name] = train_full(model, X_tr_t, y_tr_t, X_val_t, y_val_t, epochs=EPOCHS)
    final_val_acc = results[name]['val_acc'][-1]
    print(f'val_acc final: {final_val_acc:.3f}')

In [ ]:
# ─── Visualização: 3 painéis diagnósticos ─────────────────────────────────────
# O shape das curvas é o diagnóstico — não a métrica final isolada.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Diagnóstico arquitetural — Capacidade vs. Generalização', fontsize=13, y=1.01)

epochs_range = range(1, EPOCHS + 1)
color_list = list(COLORS.values())

for (name, h), color in zip(results.items(), color_list):
    n_params = param_counts[name]
    label = f'{name} ({n_params:,} params)'

    # Painel 1 — Loss treino vs validação
    axes[0].plot(epochs_range, h['tr_loss'],  color=color, lw=2,   label=f'{name} treino')
    axes[0].plot(epochs_range, h['val_loss'], color=color, lw=1.5, linestyle='--', alpha=0.7, label=f'{name} val')

    # Painel 2 — Accuracy de validação
    axes[1].plot(epochs_range, [v * 100 for v in h['val_acc']], color=color, lw=2, label=label)

    # Painel 3 — Gap val_loss − tr_loss
    # Gap crescente = sinal de alarme arquitetural.
    # Gap estável (mesmo positivo) = inductive bias adequado para o volume de dados.
    axes[2].plot(epochs_range, h['gap'], color=color, lw=2, label=name)

# Formatação
axes[0].set_title('Loss: treino vs validação')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Cross-entropy loss')
axes[0].legend(ncol=1, fontsize=8)

axes[1].set_title('Accuracy de validação')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 105)
axes[1].legend()

axes[2].set_title('Gap val/treino — diagnóstico de overfitting')
axes[2].set_xlabel('Época')
axes[2].set_ylabel('val_loss − tr_loss')
axes[2].axhline(0, color='gray', lw=0.8, linestyle=':')
axes[2].fill_between(epochs_range,
                     results['MLP Largo']['gap'],
                     alpha=0.07, color=COLORS['mlp_largo'],
                     label='área de overfitting (MLP Largo)')
axes[2].legend()

plt.tight_layout()
plt.savefig('diagnostico_capacidade.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: diagnostico_capacidade.png')

In [ ]:
# ─── Tabela de diagnóstico final ─────────────────────────────────────────────
# Como ler o diagnóstico:
#   tr_acc alto + val_acc baixo  → overfitting estrutural (capacidade > volume de dados)
#   tr_acc baixo + val_acc baixo → underfitting (capacidade insuficiente)
#   gap crescendo com épocas     → a rede está memorizando, não generalizando

print('\n' + '=' * 70)
print(f'{"Modelo":<15} {"Params":>10} {"tr_acc":>8} {"val_acc":>8} {"gap":>8} {"Diagnóstico"}')
print('=' * 70)

for name, h in results.items():
    tr_acc  = h['tr_acc'][-1]
    val_acc = h['val_acc'][-1]
    gap     = h['gap'][-1]
    n_params = param_counts[name]

    if tr_acc < 0.70:
        diag = '⚠️ Underfitting — falta capacidade'
    elif gap > 0.40:
        diag = '🔴 Overfitting estrutural — excesso de capacidade'
    elif gap > 0.15:
        diag = '🟡 Gap moderado — monitorar'
    else:
        diag = '✅ Capacidade adequada'

    print(f'{name:<15} {n_params:>10,} {tr_acc:>7.1%} {val_acc:>7.1%} {gap:>8.3f}  {diag}')

print('=' * 70)

---

## Experimento 2 — Fórmula de Parâmetros por Camada

**Fórmula:**

$$P = \sum_{l=1}^{L} \left( n_{l-1} \cdot n_l + n_l \right)$$

Onde:
- $n_l$ = número de unidades na camada $l$
- $n_{l-1} \cdot n_l$ = parâmetros de peso (conexões)
- $n_l$ = parâmetros de bias

**O gráfico abaixo mostra onde a capacidade está concentrada em cada arquitetura.**

In [ ]:
def param_breakdown(layer_sizes, input_dim=64, n_classes=10):
    """
    Calcula P = Σ(n_{l-1} · n_l + n_l) por camada.
    Devolve lista de (nome_camada, n_params, pct_do_total).

    Por que auditar por camada e não só o total?
    Em redes grandes, a primeira camada costuma dominar o total de parâmetros
    (ex: 784 → 512 = 401.920 só de pesos). Se você está com overfitting,
    saber que 70% dos parâmetros estão na camada 1 guia onde reduzir.
    """
    sizes = [input_dim] + list(layer_sizes) + [n_classes]
    breakdown = []
    for i in range(1, len(sizes)):
        n_weights = sizes[i - 1] * sizes[i]
        n_bias    = sizes[i]
        total     = n_weights + n_bias
        name = f'Camada {i}' if i < len(sizes) - 1 else 'Saída'
        breakdown.append({
            'camada':   name,
            'entrada':  sizes[i - 1],
            'saída':    sizes[i],
            'pesos':    n_weights,
            'bias':     n_bias,
            'total':    total,
        })

    total_geral = sum(d['total'] for d in breakdown)
    for d in breakdown:
        d['pct'] = d['total'] / total_geral * 100
    return breakdown, total_geral


arch_configs = {
    'MLP Raso':  [32],
    'MLP Médio': [256, 128],
    'MLP Largo': [512, 256],
}

print('\nParâmetros por camada — P = Σ(n_{l-1} · n_l + n_l)\n')
print(f'{"Arquitetura":<12} {"Camada":<10} {"Entrada":>8} {"Saída":>8} {"Pesos":>10} {"Bias":>6} {"Total":>10} {"% do total":>10}')
print('-' * 80)

all_breakdowns = {}
for name, sizes in arch_configs.items():
    bd, total = param_breakdown(sizes)
    all_breakdowns[name] = (bd, total)
    for row in bd:
        print(f'{name:<12} {row["camada"]:<10} {row["entrada"]:>8} {row["saída"]:>8} '
              f'{row["pesos"]:>10,} {row["bias"]:>6,} {row["total"]:>10,} {row["pct"]:>9.1f}%')
    print(f'{"":<12} {"TOTAL":<10} {"":<8} {"":<8} {"":<10} {"":<6} {total:>10,}  100.0%')
    print()

In [ ]:
# ─── Gráfico: distribuição de parâmetros por camada ───────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Distribuição de parâmetros por camada\nP = Σ(n_{l−1} · n_l + n_l)', fontsize=12, y=1.02)

for ax, (name, (bd, total)), color in zip(axes, all_breakdowns.items(), color_list):
    camadas = [d['camada'] for d in bd]
    pcts    = [d['pct']    for d in bd]
    totais  = [d['total']  for d in bd]

    bars = ax.bar(camadas, pcts, color=color, alpha=0.85, edgecolor='white', linewidth=0.8)

    # Anotação: parâmetros absolutos em cada barra
    for bar, t in zip(bars, totais):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f'{t:,}',
            ha='center', va='bottom', fontsize=8, color='#333'
        )

    ax.set_title(f'{name}\n({total:,} params total)', fontsize=10)
    ax.set_xlabel('Camada')
    ax.set_ylabel('% dos parâmetros totais')
    ax.set_ylim(0, 105)
    ax.tick_params(axis='x', rotation=15)

    # Insight: se uma camada domina >70%, cheque se ela é realmente necessária
    max_pct = max(pcts)
    if max_pct > 70:
        ax.annotate(
            f'⚠️ {max_pct:.0f}% dos params\nnuma camada',
            xy=(pcts.index(max_pct), max_pct),
            xytext=(pcts.index(max_pct) + 0.15, max_pct - 15),
            fontsize=8, color='#c0592a',
            arrowprops=dict(arrowstyle='->', color='#c0592a', lw=1)
        )

plt.tight_layout()
plt.savefig('params_por_camada.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: params_por_camada.png')

---

## Experimento 3 — Inductive Bias por Tipo de Dado

O critério de decisão não é benchmark — é **alinhamento entre inductive bias e estrutura do problema**.

| Arquitetura | Inductive bias | Quando usar |
|---|---|---|
| MLP | Todas features interagem com todas | Interações densas, estrutura desconhecida |
| CNN | Localidade, equivariância à translação | Dados espaciais (imagem, sinal) |
| LSTM | Dependência sequencial ordenada | Séries com dependência temporal |
| Transformer | Relações pairwise aprendidas | Dependências não-locais |

**Critério prático de decisão:**
- Erro de treino alto, val próximo → underfitting → adicione capacidade
- Erro de treino baixo, val alto → overfitting → adicione regularização ou reduza a rede
- Gap crescendo com épocas → inductive bias inadequado ou capacidade excessiva

In [ ]:
# ─── Teste do critério prático: o que acontece ao variar a capacidade ─────────
# Para cada nível de capacidade, treina e mede o gap.
# Isso torna o critério de decisão empiricamente verificável.

def scan_capacity(hidden_sizes_list, X_tr, y_tr, X_val, y_val, epochs=25):
    """
    Testa várias configurações de rede e devolve parâmetros, val_acc e gap final.
    Útil para encontrar o ponto de transição underfitting → adequado → overfitting.
    """
    scan_results = []
    for sizes in hidden_sizes_list:
        layers = []
        in_dim = X_tr.shape[1]
        for h in sizes:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, 10))
        model = nn.Sequential(*layers)

        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        h = train_full(model, X_tr, y_tr, X_val, y_val, epochs=epochs, lr=1e-3)

        scan_results.append({
            'config':   str(sizes),
            'n_params': n_params,
            'val_acc':  h['val_acc'][-1],
            'gap':      h['gap'][-1],
            'tr_acc':   h['tr_acc'][-1],
        })
        print(f'  {str(sizes):<20} params={n_params:>8,} | val_acc={h["val_acc"][-1]:.3f} | gap={h["gap"][-1]:.3f}')

    return scan_results


hidden_configs = [
    [8],
    [16],
    [32],
    [64],
    [128],
    [256],
    [512],
    [64, 32],
    [128, 64],
    [256, 128],
    [512, 256],
    [512, 256, 128],
]

print('Scanning capacidades — de underfitting a overfitting:')
scan = scan_capacity(hidden_configs, X_tr_t, y_tr_t, X_val_t, y_val_t, epochs=30)

In [ ]:
# ─── Gráfico: val_acc e gap vs. número de parâmetros ─────────────────────────
# Esse gráfico é o argumento visual central do framework de decisão:
# existe um ponto de transição onde mais capacidade não ajuda a generalização
# e começa a aumentar o gap (sinal de memorização).

n_params_list = [r['n_params'] for r in scan]
val_acc_list  = [r['val_acc']  * 100 for r in scan]
gap_list      = [r['gap']      for r in scan]
configs       = [r['config']   for r in scan]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Capacidade vs. Generalização\nO ponto de transição entre underfitting e overfitting', fontsize=12)

# Painel superior: val_acc
ax1.plot(range(len(scan)), val_acc_list, color=COLORS['mlp_medio'],
         lw=2, marker='o', markersize=6)

# Marca o ponto ótimo
best_idx = val_acc_list.index(max(val_acc_list))
ax1.axvline(best_idx, color='green', lw=1, linestyle='--', alpha=0.6)
ax1.scatter([best_idx], [val_acc_list[best_idx]], color='green', zorder=5, s=80)
ax1.annotate(
    f'Ponto ótimo\n{configs[best_idx]}\n{n_params_list[best_idx]:,} params',
    xy=(best_idx, val_acc_list[best_idx]),
    xytext=(best_idx + 0.5, val_acc_list[best_idx] - 8),
    fontsize=8, color='green',
    arrowprops=dict(arrowstyle='->', color='green', lw=1)
)
ax1.set_ylabel('Val accuracy (%)')
ax1.set_ylim(0, 105)

# Painel inferior: gap
ax2.bar(range(len(scan)), gap_list,
        color=[COLORS['mlp_raso'] if g > 0.3 else COLORS['mlp_medio'] if g > 0.1 else '#888'
               for g in gap_list],
        alpha=0.8, edgecolor='white')
ax2.axhline(0.1, color='orange', lw=1, linestyle='--', alpha=0.7, label='gap moderado (0.10)')
ax2.axhline(0.3, color='red',    lw=1, linestyle='--', alpha=0.7, label='gap alto (0.30)')
ax2.axvline(best_idx, color='green', lw=1, linestyle='--', alpha=0.6)
ax2.set_ylabel('Gap val/treino')
ax2.set_xlabel('Configuração de rede')
ax2.set_xticks(range(len(scan)))
ax2.set_xticklabels(configs, rotation=45, ha='right', fontsize=8)
ax2.legend(fontsize=8)

# Anotação de regiões
ax1.axvspan(0, best_idx - 0.5, alpha=0.05, color='orange', label='underfitting')
ax1.axvspan(best_idx - 0.5, best_idx + 0.5, alpha=0.05, color='green', label='zona adequada')
ax1.axvspan(best_idx + 0.5, len(scan) - 0.5, alpha=0.05, color='red', label='overfitting')
ax1.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig('capacidade_vs_generalizacao.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: capacidade_vs_generalizacao.png')

---

## Resumo: Critério de Decisão Arquitetural

Use a tabela abaixo como guia antes de redesenhar qualquer arquitetura.

In [ ]:
# ─── Diagnóstico automático a partir das métricas ────────────────────────────

def diagnostico(tr_loss, val_loss, tr_acc, val_acc, epoch_trend_gap=None):
    """
    Devolve diagnóstico e ação recomendada com base nas métricas finais.

    Baseado no critério do episódio:
    - tr_acc baixo + val_acc baixo  → underfitting (capacidade)
    - tr_acc alto  + val_acc baixo  → overfitting  (capacidade ou inductive bias)
    - gap crescendo ao longo do treino → overfitting estrutural
    """
    gap = val_loss - tr_loss

    if tr_acc < 0.75:
        return {
            'status': '⚠️ Underfitting',
            'causa': 'Capacidade insuficiente para o problema',
            'acao': 'Adicione largura (neurônios) ou profundidade (camadas).\n'
                    'NÃO adicione regularização — o problema não é overfitting.',
        }
    elif gap > 0.40:
        return {
            'status': '🔴 Overfitting estrutural',
            'causa': 'Capacidade excessiva para o volume de dados ou inductive bias desalinhado',
            'acao': 'Opções em ordem de preferência:\n'
                    '  1. Reduza a capacidade (menos neurônios/camadas)\n'
                    '  2. Adicione Dropout ou weight decay\n'
                    '  3. Verifique se o inductive bias faz sentido para o problema\n'
                    '  NÃO adicione mais camadas.',
        }
    elif gap > 0.15:
        return {
            'status': '🟡 Gap moderado',
            'causa': 'Overfitting leve — aceitável dependendo do domínio',
            'acao': 'Monitore se o gap cresce com mais épocas.\n'
                    'Considere early stopping ou leve regularização.',
        }
    else:
        return {
            'status': '✅ Capacidade adequada',
            'causa': 'Inductive bias e capacidade alinhados ao volume de dados',
            'acao': 'Proceda para ajuste fino de hiperparâmetros e validação em produção.',
        }


print('\nDIAGNÓSTICO FINAL — modelos treinados neste notebook')
print('=' * 65)

for name, h in results.items():
    tr_loss  = h['tr_loss'][-1]
    val_loss = h['val_loss'][-1]
    tr_acc   = h['tr_acc'][-1]
    val_acc  = h['val_acc'][-1]

    diag = diagnostico(tr_loss, val_loss, tr_acc, val_acc)

    print(f'\n{name} ({param_counts[name]:,} params)')
    print(f'  tr_acc={tr_acc:.3f} | val_acc={val_acc:.3f} | gap={val_loss - tr_loss:.3f}')
    print(f'  Status:  {diag["status"]}')
    print(f'  Causa:   {diag["causa"]}')
    print(f'  Ação:    {diag["acao"]}')

print('\n' + '=' * 65)
print('\nArquivos gerados:')
print('  diagnostico_capacidade.png')
print('  params_por_camada.png')
print('  capacidade_vs_generalizacao.png')

---

## Próximos passos

Este notebook cobre os Atos 4, 5 e 6 do episódio.  
Para o **Ato 7 (produção)**, o repositório do canal inclui:

- Pipeline com Fashion-MNIST (28×28, mais próximo de casos reais de imagem)
- Ablation estruturado com **Banking77** (intent classification, 77 classes) — demonstra diferença MLP vs arquitetura com inductive bias linguístico
- Rastreamento de experimentos com **MLflow** — registra hipótese arquitetural junto com métricas
- Early stopping automático baseado no critério de gap

```
github.com/airoots/neural-arch-decision
```

---

**AI Roots** — Decisão arquitetural é derivação de hipótese, não busca aleatória.